# Semana 9 - Optimización de un Árbol de Decisión con Diabetes

**Materia:** Minería de Datos  
**Tema:** Optimización mediante GridSearchCV  
**Estudiante:** Sara Valenzuela

## Objetivo

Entrenar un árbol de decisión utilizando el dataset Diabetes y optimizar sus hiperparámetros mediante GridSearchCV para mejorar el desempeño del modelo.

In [1]:
import pandas as pd
import numpy as np

from sklearn.datasets import load_diabetes
from sklearn.model_selection import train_test_split
from sklearn.model_selection import GridSearchCV

from sklearn.tree import DecisionTreeClassifier

from sklearn.metrics import (
    accuracy_score,
    classification_report
)

In [2]:
# Cargar dataset diabetes
diabetes = load_diabetes()

# Convertir a DataFrame
df = pd.DataFrame(
    diabetes.data,
    columns=diabetes.feature_names
)

# Variable objetivo
df["target"] = diabetes.target

df.head()

,age,sex,bmi,bp,s1,s2,s3,s4,s5,s6,target
0,0.038076,0.050680,0.061696,0.021872,-0.044223,-0.034821,-0.043401,-0.002592,0.019907,-0.017646,151.0
1,-0.001882,-0.044642,-0.051474,-0.026328,-0.008449,-0.019163,0.074412,-0.039493,-0.068332,-0.092204,75.0
2,0.085299,0.050680,0.044451,-0.005670,-0.045599,-0.034194,-0.032356,-0.002592,0.002861,-0.025930,141.0
3,-0.089063,-0.044642,-0.011595,-0.036656,0.012191,0.024991,-0.036038,0.034309,0.022688,-0.009362,206.0
4,0.005383,-0.044642,-0.036385,0.021872,0.003935,0.015596,0.008142,-0.002592,-0.031988,-0.046641,135.0


In [3]:
# Crear clases Alto / Bajo usando la mediana

mediana = df["target"].median()

df["diagnostico"] = np.where(
    df["target"] >= mediana,
    "Alto",
    "Bajo"
)

df.head()

,age,sex,bmi,bp,s1,s2,s3,s4,s5,s6,target,diagnostico
0,0.038076,0.050680,0.061696,0.021872,-0.044223,-0.034821,-0.043401,-0.002592,0.019907,-0.017646,151.0,Alto
1,-0.001882,-0.044642,-0.051474,-0.026328,-0.008449,-0.019163,0.074412,-0.039493,-0.068332,-0.092204,75.0,Bajo
2,0.085299,0.050680,0.044451,-0.005670,-0.045599,-0.034194,-0.032356,-0.002592,0.002861,-0.025930,141.0,Alto
3,-0.089063,-0.044642,-0.011595,-0.036656,0.012191,0.024991,-0.036038,0.034309,0.022688,-0.009362,206.0,Alto
4,0.005383,-0.044642,-0.036385,0.021872,0.003935,0.015596,0.008142,-0.002592,-0.031988,-0.046641,135.0,Bajo


## Preparación de los datos

El dataset Diabetes posee originalmente una variable objetivo numérica.

Para poder aplicar un árbol de clasificación, se transformó la variable objetivo en dos categorías: "Alto" y "Bajo", utilizando la mediana como punto de corte.

In [4]:
X = df.drop(
    columns=["target", "diagnostico"]
)

y = df["diagnostico"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)

print(X_train.shape)
print(X_test.shape)

(353, 10)
(89, 10)


In [5]:
modelo_base = DecisionTreeClassifier(
    random_state=42
)

modelo_base.fit(
    X_train,
    y_train
)

pred_base = modelo_base.predict(
    X_test
)

acc_base = accuracy_score(
    y_test,
    pred_base
)

print("Accuracy Base:", acc_base)

Accuracy Base: 0.6741573033707865


In [6]:
param_grid = {
    "max_depth": [2, 3, 4, 5, 6],
    "criterion": ["gini", "entropy"],
    "min_samples_split": [2, 5, 10]
}

grid = GridSearchCV(
    DecisionTreeClassifier(random_state=42),
    param_grid,
    cv=5,
    scoring="accuracy"
)

grid.fit(
    X_train,
    y_train
)

print("Mejores parámetros:")
print(grid.best_params_)

Mejores parámetros:
{'criterion': 'gini', 'max_depth': 2, 'min_samples_split': 2}


In [7]:
mejor_modelo = grid.best_estimator_

pred_opt = mejor_modelo.predict(
    X_test
)

acc_opt = accuracy_score(
    y_test,
    pred_opt
)

print("Accuracy Optimizado:", acc_opt)

Accuracy Optimizado: 0.7303370786516854


In [8]:
print(
    classification_report(
        y_test,
        pred_opt
    )
)

              precision    recall  f1-score   support

        Alto       0.65      0.88      0.74        40
        Bajo       0.86      0.61      0.71        49

    accuracy                           0.73        89
   macro avg       0.75      0.74      0.73        89
weighted avg       0.76      0.73      0.73        89



## Conclusión Final

Se utilizó el dataset Diabetes para construir un modelo de clasificación mediante árboles de decisión. Debido a que la variable objetivo original es numérica, se transformó en dos categorías ("Alto" y "Bajo") utilizando la mediana como punto de corte.

El modelo base obtuvo una precisión de 67,42%, mientras que el modelo optimizado mediante GridSearchCV alcanzó una precisión de 73,03%.

La búsqueda automática de hiperparámetros permitió identificar como mejor configuración un árbol con criterio Gini, profundidad máxima de 2 niveles y un mínimo de 2 observaciones para realizar divisiones internas.

Los resultados evidencian que la optimización mejoró el rendimiento del modelo, aumentando su capacidad de clasificación sobre datos no observados. Esto demuestra la importancia de ajustar adecuadamente los hiperparámetros para obtener modelos más robustos y con mejor capacidad de generalización.